# Threat intelligence feed analysis (OTX snapshot, 2025-09-25)

**Question:** what does a subscription to AlienVault OTX actually deliver, and what should an analyst watch out for before using it?

**Data:** `data/raw_iocs.csv`, 30,343 indicators collected by this project's collector from the OTX *subscribed pulses* endpoint on 2025-09-25. A *pulse* is a published threat report; each pulse lists indicators (domains, IPs, file hashes, URLs, CVEs).

Every number below is computed in this notebook. Re-run it top to bottom to reproduce.

In [1]:
import sys; sys.path.insert(0, "..")  # run from the notebooks/ folder
from pathlib import Path
import pandas as pd
from threat_collector import analyze

FIG = Path("../docs/figures")

def save_and_show(svg, name):
    (FIG / name).write_text(svg, encoding="utf-8")
    try:
        from IPython.display import SVG, display
        display(SVG(svg))
    except ImportError:
        print(f"saved {FIG / name}")

df = pd.read_csv("../data/raw_iocs.csv")
print(f"{len(df):,} indicators, {df['pulse_name'].nunique()} pulses, {df['type'].nunique()} indicator types")

30,343 indicators, 49 pulses, 8 indicator types


## 1. Data quality

Before analysing, check duplicates and whether values have the format their type claims.

In [2]:
q = analyze.quality_checks(df)
q

{'rows': 30343, 'duplicate_values': 56, 'duplicates_within_same_pulse': 53, 'missing_values': 0, 'invalid_format': {'FileHash-MD5': 0, 'FileHash-SHA1': 0, 'FileHash-SHA256': 0, 'IPv4': 0}}

## 2. How concentrated is the feed?

In [3]:
c = analyze.pulse_concentration(df)
print(f"Median pulse: {c['median_per_pulse']:.0f} indicators")
print(f"Largest pulse: '{c['largest_pulse']}' = {c['largest_share']:.1%} of all indicators")
print(f"Top 3 pulses = {c['top3_share']:.1%}")
iofi = df["pulse_name"].str.startswith("Infrastructure of Interest")
print(f"The {df.loc[iofi, 'pulse_name'].nunique()} 'Infrastructure of Interest' pulses = {iofi.mean():.1%}")

Median pulse: 25 indicators
Largest pulse: 'Infrastructure of Interest: Medium Confidence Detection' = 79.4% of all indicators
Top 3 pulses = 91.5%
The 4 'Infrastructure of Interest' pulses = 91.5%


In [4]:
top = c["counts"].head(10)
short = lambda n: n.replace("Infrastructure of Interest: Medium Confidence ", "Infra. of Interest: ")
labels = [short(n) if len(short(n)) <= 42 else short(n)[:40] + "..." for n in top.index]
svg = analyze.svg_bar_panels(
    [{"labels": labels, "values": list(top.values),
      "muted": {l for l, n in zip(labels, top.index) if not n.startswith("Infrastructure of Interest")}}],
    title="One pulse supplies 79% of all indicators",
    subtitle="Top 10 of 49 pulses by indicator count; violet = 'Infrastructure of Interest' feeds",
    label_width=320, panel_width=360)
save_and_show(svg, "pulse_concentration.svg")

saved ../docs/figures/pulse_concentration.svg


## 3. Does that one pulse distort the picture?

Compare the indicator-type mix with and without the largest pulse.

In [5]:
big = c["largest_pulse"]
mix = pd.DataFrame({
    "All pulses": analyze.type_share(df),
    "Without largest pulse": analyze.type_share(df[df["pulse_name"] != big]),
}).fillna(0).sort_values("Without largest pulse")
(mix * 100).round(1).iloc[::-1]

                 All pulses  Without largest pulse
type                                              
domain                 91.1                   60.2
FileHash-SHA256         2.9                   14.0
hostname                2.3                    7.5
FileHash-MD5            1.3                    6.3
FileHash-SHA1           1.2                    5.7
IPv4                    0.7                    3.2
URL                     0.5                    2.5
CVE                     0.1                    0.6

In [6]:
order = mix.sort_values("All pulses", ascending=False).index
svg = analyze.svg_bar_panels(
    [{"title": col, "labels": list(order), "values": list((mix.loc[order, col] * 100).round(1))}
     for col in mix.columns],
    title="Without the largest pulse, file hashes are a quarter of the feed",
    subtitle="Share of indicators by type (%)", fmt="{:.1f}%", label_width=150, panel_width=300)
save_and_show(svg, "type_mix.svg")

saved ../docs/figures/type_mix.svg


## 4. Which top-level domains do reported domains use?

In [7]:
share, n_tld = analyze.tld_share(df, top=10)
rest, _ = analyze.tld_share(df[df["pulse_name"] != big], top=5)
print(f"{n_tld} distinct top-level domains")
print("Top 5 without the largest pulse:", (rest * 100).round(1).to_dict())
(share * 100).round(1)

252 distinct top-level domains
Top 5 without the largest pulse: {'online': 22.2, 'store': 15.3, 'site': 11.8, 'com': 6.6, 'shop': 6.3}


0
online    42.2
site      21.7
xyz       11.9
com        4.5
top        2.8
store      2.6
bond       1.7
org        1.5
shop       1.3
click      1.1
Name: proportion, dtype: float64

In [8]:
svg = analyze.svg_bar_panels(
    [{"labels": ["." + t for t in share.index], "values": list((share * 100).round(1)), "muted": {".com"}}],
    title="Three TLDs hold 76% of reported domains; .com only 4.5%",
    subtitle="Share of domains and hostnames by top-level domain (%)", fmt="{:.1f}%",
    label_width=90, panel_width=520)
save_and_show(svg, "tld_share.svg")

saved ../docs/figures/tld_share.svg


## 5. CVEs referenced

In [9]:
cves = df.loc[df["type"] == "CVE", "indicator"]
cves.str.extract(r"CVE-(\d{4})")[0].value_counts().sort_index()

0
2013     3
2017     1
2020     2
2022     2
2023     3
2024    14
2025    11
Name: count, dtype: int64

## Findings

See [`REPORT.md`](../REPORT.md) for the written findings and their limitations.